# 9.3 Boosting

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.03_boosting.ipynb)

Companion code for Section 9.3 of the book. Run the cells in order from top to bottom.

**Example 9.3.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

# ── Gradient Boosting from scratch (simplified) ───────────────────
class GradientBoostingScratch:
    '''Gradient Boosting for binary classification.'''

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators  = n_estimators
        self.lr            = learning_rate
        self.max_depth     = max_depth
        self.trees_        = []
        self.F0_           = None
        self.train_losses_ = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def _log_loss(self, y, p):
        p = np.clip(p, 1e-10, 1-1e-10)
        return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

    def fit(self, X, y):
        from sklearn.tree import DecisionTreeRegressor

        # F₀: log-odds of positive class
        p0 = y.mean()
        self.F0_ = np.log(p0 / (1 - p0 + 1e-10))
        F = np.full(len(y), self.F0_)

        for m in range(self.n_estimators):
            # Pseudo-residuals for log-loss: y - sigmoid(F)
            p = self._sigmoid(F)
            residuals = y - p

            # Fit tree to pseudo-residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees_.append(tree)

            # Update F
            F += self.lr * tree.predict(X)

            # Track loss
            loss = self._log_loss(y, self._sigmoid(F))
            self.train_losses_.append(loss)

        return self

    def predict_proba(self, X):
        F = np.full(len(X), self.F0_)
        for tree in self.trees_:
            F += self.lr * tree.predict(X)
        p = self._sigmoid(F)
        return np.column_stack([1-p, p])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:,1] >= threshold).astype(int)

    def score(self, X, y):
        return (self.predict(X) == y).mean()


# ── Compare bagging vs boosting ───────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Gradient Boosting from scratch
gb_scratch = GradientBoostingScratch(n_estimators=100, learning_rate=0.1, max_depth=3)
gb_scratch.fit(X_tr, y_tr)
print(f'From-scratch GB: {gb_scratch.score(X_te, y_te):.4f}')

# Sklearn comparison
print('\n=== Bagging vs Boosting Comparison ===')
print(f'{'Method':<28} {'n':>5} {'Train':>8} {'Test':>8} {'CV(5)':>8}')
print('='*62)

from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

methods = [
    ('Single Tree (depth=3)',   DecisionTreeClassifier(max_depth=3, random_state=42), 1),
    ('Single Tree (full)',      DecisionTreeClassifier(random_state=42), 1),
    ('Bagging (n=100)',         BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42), 100),
    ('GradBoost (lr=0.1,d=3)', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
random_state=42), 100),
    ('GradBoost (lr=0.01)',    GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3,
random_state=42), 500),
]

for name, model, n in methods:
    model.fit(X_tr, y_tr)
    tr  = model.score(X_tr, y_tr)
    te  = model.score(X_te, y_te)
    cv  = cross_val_score(clone(model), X, y, cv=5, scoring='accuracy').mean()
    print(f'{name:<28} {n:>5} {tr:>8.4f} {te:>8.4f} {cv:>8.4f}')

# ── Visualise: training loss curve for GB ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Boosting Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(gb_scratch.train_losses_, 'b-', lw=2)
ax.set_xlabel('Boosting Rounds'); ax.set_ylabel('Log-Loss')
ax.set_title('From-Scratch GB: Training Loss\n(each round corrects previous errors)')
ax.grid(alpha=0.3)

# Staged predictions
ax = axes[1]
gb_staged = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                         max_depth=3, random_state=42)
gb_staged.fit(X_tr, y_tr)
train_staged = list(gb_staged.staged_score(X_tr, y_tr))
test_staged  = list(gb_staged.staged_score(X_te, y_te))
rounds = range(1, 201)
ax.plot(rounds, train_staged, 'b-', lw=2, alpha=0.8, label='Train')
ax.plot(rounds, test_staged,  'r-', lw=2, alpha=0.8, label='Test')
best_round = np.argmax(test_staged) + 1
ax.axvline(best_round, color='green', ls='--', lw=2,
          label=f'Best round={best_round} (acc={max(test_staged):.4f})')
ax.set_xlabel('Boosting Rounds'); ax.set_ylabel('Accuracy')
ax.set_title('Staged Accuracy: Train vs Test\n(test peaks then slightly decreases)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('boosting_analysis.png', dpi=150, bbox_inches='tight')
plt.show()